# Exploratory Data Analysis (EDA)
This notebook guides you through the Exploratory Data Analysis phase of the Predictive Analytics project. We cover data loading, automatic type detection, cleaning, univariate/bivariate distributions, correlation analysis, and time-series trend analysis.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

### 1. Load Data
We will load our default stock market or synthetic dataset. If the dataset does not exist, we will use our helper to download/generate it.

In [ ]:
from src.data_preprocessing import load_data, detect_dataset_type, generate_default_timeseries

data_path = '../data/raw/default_stock_data.csv'
if not os.path.exists(data_path):
    df_raw = generate_default_timeseries(data_path)
else:
    df_raw = load_data(data_path)

print(f"Dataset shape: {df_raw.shape}")
df_raw.head()

### 2. Auto-Detect Dataset Type
We check if the dataset is Time Series or standard tabular Regression based on date columns.

In [ ]:
dataset_type, datetime_col = detect_dataset_type(df_raw)
print(f"Detected Dataset Type: {dataset_type.upper()}")
print(f"Detected Datetime Column: {datetime_col}")

### 3. Data Cleaning
Clean the dataset by removing duplicates, handling missing values, and capping outliers.

In [ ]:
from src.data_preprocessing import clean_data

df_cleaned = clean_data(df_raw, datetime_col=datetime_col)
print(f"Cleaned Dataset shape: {df_cleaned.shape}")
df_cleaned.head()

### 4. Univariate Analysis
Examine distributions of numeric features using histograms and boxplots.

In [ ]:
# Find a numeric column to plot (e.g. target Close or Sales)
target_candidates = ['Close', 'Sales', df_cleaned.columns[0]]
target_col = next((c for c in target_candidates if c in df_cleaned.columns), df_cleaned.columns[0])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Histogram & Density
sns.histplot(df_cleaned[target_col], kde=True, ax=axes[0], color='royalblue')
axes[0].set_title(f'Distribution of {target_col}')

# Boxplot to see outliers
sns.boxplot(y=df_cleaned[target_col], ax=axes[1], color='coral')
axes[1].set_title(f'Boxplot of {target_col}')
plt.tight_layout()
plt.show()

### 5. Bivariate Analysis
Examine relationships between columns. Plot correlation heatmap and scatter plots.

In [ ]:
# Heatmap of correlations
numeric_df = df_cleaned.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

### 6. Time Series Specific Analysis (Trend & Rolling Stats)
If dataset is Time Series, plot trend and rolling averages.

In [ ]:
if dataset_type == 'time_series':
    df_cleaned['Rolling_Mean_30'] = df_cleaned[target_col].rolling(window=30).mean()
    df_cleaned['Rolling_Std_30'] = df_cleaned[target_col].rolling(window=30).std()
    
    plt.figure(figsize=(14, 7))
    plt.plot(df_cleaned[target_col], label='Original Close Price', alpha=0.5)
    plt.plot(df_cleaned['Rolling_Mean_30'], label='30-Day Rolling Mean', color='red', linewidth=2)
    plt.plot(df_cleaned['Rolling_Std_30'], label='30-Day Rolling Std', color='green', linestyle='--')
    plt.title(f'{target_col} Price Trend & Rolling Statistics')
    plt.legend()
    plt.show()
else:
    print("Dataset is classified as standard Regression, skipping time-series rolling plots.")